<a href="https://colab.research.google.com/github/SVz54/9517/blob/draft2/9517.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:

# --- Kaggle download (same as before) ---
import os, json, shutil, zipfile, glob, pathlib

os.makedirs('/root/.kaggle', exist_ok=True)
if not os.path.exists('/root/.kaggle/kaggle.json') and os.path.exists('/content/kaggle.json'):
    shutil.move('/content/kaggle.json', '/root/.kaggle/kaggle.json')
!chmod 600 /root/.kaggle/kaggle.json

!pip -q install kaggle
DATA_DIR = "/content/AgroPest12"
os.makedirs(DATA_DIR, exist_ok=True)
!kaggle datasets download -d rupankarmajumdar/crop-pests-dataset -p $DATA_DIR -q

# Unzip (overwrite if re-running)
zip_files = glob.glob(f"{DATA_DIR}/*.zip")
assert zip_files, "Zip not found – did the Kaggle download succeed?"
zip_path = zip_files[0]
!unzip -q -o "$zip_path" -d "$DATA_DIR"

# --- Auto-detect BASE: the folder that contains train/valid/test with images+labels ---
def find_yolo_base(root):
    for p, d, f in os.walk(root):
        if (os.path.isdir(os.path.join(p, "train", "images")) and
            os.path.isdir(os.path.join(p, "train", "labels")) and
            os.path.isdir(os.path.join(p, "valid", "images")) and
            os.path.isdir(os.path.join(p, "valid", "labels"))):
            return p
    return None

BASE = find_yolo_base(DATA_DIR)
assert BASE is not None, f"Could not find YOLO folders under {DATA_DIR}. Found: {os.listdir(DATA_DIR)}"
print("BASE:", BASE)
print("train samples:", len(glob.glob(os.path.join(BASE, "train/images/*.jpg"))))
print("val samples:", len(glob.glob(os.path.join(BASE, "valid/images/*.jpg"))))
print("test samples:", len(glob.glob(os.path.join(BASE, "test/images/*.jpg"))))
BASE = pathlib.Path(BASE)  # keep as Path for later cells

Dataset URL: https://www.kaggle.com/datasets/rupankarmajumdar/crop-pests-dataset
License(s): MIT
BASE: /content/AgroPest12
train samples: 11502
val samples: 1095
test samples: 546


In [4]:
!pip -q install --upgrade ultralytics==8.3.20 opencv-python-headless==4.10.0.84 matplotlib==3.9.2


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 876.6/876.6 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 55.4 MB/s eta 0:00:00


In [5]:
from pathlib import Path
yaml_path = BASE / "data.yaml"
txt = f"""# AgroPest-12
path: {BASE.as_posix()}
train: train/images
val: valid/images
test: test/images
names:
  0: aphid
  1: armyworm
  2: beetle
  3: bollworm
  4: grasshopper
  5: leafhopper
  6: locust
  7: mealybug
  8: mosquito
  9: moth
  10: sawfly
  11: weevil
"""
yaml_path.write_text(txt)
print("Wrote:", yaml_path)


Wrote: /content/AgroPest12/data.yaml


In [6]:
from ultralytics import YOLO
import os, glob

model = YOLO('yolov8n.pt')   # tiny + fast; swap to yolov8s.pt later if you want

# use a handful of val images for a demo run
val_imgs = sorted(glob.glob(str(BASE / 'valid/images/*.jpg')))[:12]
print("Demo images:", len(val_imgs))

pred_root = "/content/preds"
res = model.predict(val_imgs, conf=0.25, save=True, project=pred_root, name="yolo_preds", exist_ok=True, imgsz=640)
print("Saved predicted images to:", pred_root + "/yolo_preds")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


100%|██████████| 6.25M/6.25M [00:00<00:00, 125MB/s]


Demo images: 12

0: 640x640 (no detections), 5.3ms
1: 640x640 (no detections), 5.3ms
2: 640x640 1 bear, 5.3ms
3: 640x640 (no detections), 5.3ms
4: 640x640 1 cat, 5.3ms
5: 640x640 1 horse, 5.3ms
6: 640x640 (no detections), 5.3ms
7: 640x640 1 person, 5.3ms
8: 640x640 1 bird, 1 horse, 5.3ms
9: 640x640 1 teddy bear, 5.3ms
10: 640x640 1 bird, 5.3ms
11: 640x640 (no detections), 5.3ms
Speed: 2.6ms preprocess, 5.3ms inference, 26.5ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/preds/yolo_preds
Saved predicted images to: /content/preds/yolo_preds


In [7]:
# Draw a few GT boxes to confirm labels align with images
import glob, os, cv2
from pathlib import Path

samples = sorted(glob.glob(str(BASE / "valid/images/*.jpg")))[:6]
out_dir = "/content/gt_preview"; os.makedirs(out_dir, exist_ok=True)

def read_yolo_labels(lbl_path, W, H):
    boxes = []
    if not os.path.exists(lbl_path): return boxes
    for line in open(lbl_path).read().strip().splitlines():
        c, x, y, w, h = map(float, line.split())
        x1 = int((x - w/2) * W); y1 = int((y - h/2) * H)
        x2 = int((x + w/2) * W); y2 = int((y + h/2) * H)
        boxes.append((int(c), x1, y1, x2, y2))
    return boxes

for p in samples:
    img = cv2.imread(p); H, W = img.shape[:2]
    lbl = str(Path(p).with_suffix("").as_posix().replace("/images/","/labels/") + ".txt")
    for c, x1, y1, x2, y2 in read_yolo_labels(lbl, W, H):
        cv2.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 2)
        cv2.putText(img, str(c), (x1, max(0,y1-5)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
    cv2.imwrite(os.path.join(out_dir, os.path.basename(p)), img)
print("Wrote previews to", out_dir)


Wrote previews to /content/gt_preview


In [8]:
!pip -q uninstall -y wandb


In [9]:
# Quick warm-start training on AgroPest-12 (wandb off, clean project name)
import os, glob, torch
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

from ultralytics import YOLO

data_yaml = str(yaml_path)   # from your Cell 3
model = YOLO('yolov8n.pt')

results = model.train(
    data=data_yaml,
    epochs=5,                 # try 10–15 later for better boxes
    imgsz=640,
    batch=16,
    workers=2,
    lr0=0.01,
    freeze=10,
    amp=True,
    device=0 if torch.cuda.is_available() else 'cpu',
    project="agropest_runs",  # <-- no slash
    name="yolov8n_quick",
    exist_ok=True,
)

best = glob.glob("agropest_runs/yolov8n_quick/weights/best.pt")
assert best, "best.pt not found under agropest_runs/yolov8n_quick/weights/"
BEST_WEIGHTS = best[0]
print("BEST_WEIGHTS =", BEST_WEIGHTS)


New https://pypi.org/project/ultralytics/8.3.227 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.20 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/content/AgroPest12/data.yaml, epochs=5, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=2, project=agropest_runs, name=yolov8n_quick, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=10, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False

100%|██████████| 755k/755k [00:00<00:00, 28.4MB/s]


Overriding model.yaml nc=80 with nc=12

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytic

100%|██████████| 5.35M/5.35M [00:00<00:00, 111MB/s]


AMP: checks passed ✅


...

        5/5      1.36G      1.434      2.148      1.735         47        640: 100%|██████████| 719/719 [03:00<00:00,  3.98it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 35/35 [00:08<00:00,  4.21it/s]

                   all       1095       1341      0.546      0.527      0.539      0.292



5 epochs completed in 0.270 hours.
Optimizer stripped from agropest_runs/yolov8n_quick/weights/last.pt, 6.2MB
Optimizer stripped from agropest_runs/yolov8n_quick/weights/best.pt, 6.2MB

Validating agropest_runs/yolov8n_quick/weights/best.pt...
Ultralytics 8.3.20 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 168 layers, 3,007,988 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 35/35 [00:10<00:00,  3.26it/s]


                   all       1095       1341      0.545      0.528       0.54      0.292
                 aphid         96        178      0.461      0.528      0.464       0.18
              armyworm         99        110      0.575      0.555      0.547       0.25
                beetle         89        100       0.42       0.31       0.35      0.166
              bollworm         77        139      0.313      0.245      0.199     0.0883
           grasshopper         53         72      0.421      0.236      0.226        0.1
            leafhopper         91        104      0.513      0.529      0.512      0.236
                locust         98        102      0.454      0.471      0.508      0.252
              mealybug        100        101       0.81      0.802      0.898      0.581
              mosquito         77         91      0.605      0.209      0.356      0.191
                  moth         99        107      0.549      0.822      0.768      0.471
                sawfl

In [10]:
from ultralytics import YOLO
import glob, os

BEST_WEIGHTS  # should be set by previous cell
model_trained = YOLO(BEST_WEIGHTS)

val_imgs = sorted(glob.glob(str(BASE / 'valid/images/*.jpg')))[:12]
pred_root = "/content/preds"
res = model_trained.predict(
    val_imgs, conf=0.25, save=True,
    project=pred_root, name="yolo_trained", exist_ok=True, imgsz=640
)
print("Saved predicted images to:", pred_root + "/yolo_trained")



0: 640x640 (no detections), 5.6ms
1: 640x640 1 weevil, 5.6ms
2: 640x640 1 weevil, 5.6ms
3: 640x640 1 weevil, 5.6ms
4: 640x640 1 weevil, 5.6ms
5: 640x640 (no detections), 5.6ms
6: 640x640 1 weevil, 5.6ms
7: 640x640 1 weevil, 5.6ms
8: 640x640 1 weevil, 5.6ms
9: 640x640 1 weevil, 5.6ms
10: 640x640 1 weevil, 5.6ms
11: 640x640 1 weevil, 5.6ms
Speed: 2.9ms preprocess, 5.6ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/preds/yolo_trained
Saved predicted images to: /content/preds/yolo_trained


In [11]:
!pip -q uninstall -y pytorch-grad-cam grad-cam || true
!pip -q install --no-cache-dir "git+https://github.com/jacobgil/pytorch-grad-cam.git"


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [12]:
import pytorch_grad_cam, inspect
from pytorch_grad_cam.utils import model_targets
print("grad-cam version:", getattr(pytorch_grad_cam, "__version__", "git"))
print("Has YOLOv8Target:", hasattr(model_targets, "YOLOv8Target"))


grad-cam version: git
Has YOLOv8Target: False


In [16]:
# --- Run Grad-CAM on a specific image that has a good detection ---
import os, glob, cv2, torch, numpy as np, torch.nn as nn
from ultralytics import YOLO
from pytorch_grad_cam import GradCAM

BEST_WEIGHTS = "agropest_runs/yolov8n_quick/weights/best.pt"  # your trained weights
TARGET_NAME  = "Weevil-102-_jpg.rf.34c74f7621cdea247c335cac46f5c73f.jpg"  # <-- that image
INPUT_SZ     = 640
LO_PCT, HI_PCT = 75.0, 99.3
HEAT_W, IMG_W  = 0.65, 0.35
TOPK = 0.15  # paint only top 15% hottest pixels

# locate the image anywhere under BASE/**/images
cands = glob.glob(str(BASE / "**" / "images" / "*.jpg"), recursive=True)
IMG_PATH = [p for p in cands if p.endswith(TARGET_NAME)]
assert IMG_PATH, f"Couldn't find {TARGET_NAME} under {BASE}"
IMG_PATH = IMG_PATH[0]

def pct_norm(x, lo=75.0, hi=99.3, eps=1e-6):
    lo_v = np.percentile(x, lo); hi_v = np.percentile(x, hi)
    if hi_v - lo_v < 1e-8: lo_v, hi_v = x.min(), x.max() + eps
    return np.clip((x - lo_v) / (hi_v - lo_v + eps), 0, 1)

def last_conv_layer(torch_model):
    for m in list(torch_model.modules())[::-1]:
        if isinstance(m, nn.Conv2d): return m
    raise RuntimeError("No Conv2d for CAM.")

def imread_rgb(p): return cv2.cvtColor(cv2.imread(p, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
def imwrite_rgb(p, rgb): cv2.imwrite(p, cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))

# models
device = "cuda" if torch.cuda.is_available() else "cpu"
model_det = YOLO(BEST_WEIGHTS)
model_cam = YOLO(BEST_WEIGHTS)
model_cam.model.to(device).eval()
target_layers = [last_conv_layer(model_cam.model)]

# read & detect
rgb = imread_rgb(IMG_PATH)
H, W = rgb.shape[:2]
res = model_det.predict(source=rgb, conf=0.20, imgsz=INPUT_SZ, verbose=False)[0]
assert res.boxes is not None and len(res.boxes) > 0, "No detections; lower conf or increase imgsz."

# take top box
i = int(torch.argmax(res.boxes.conf).item())
x1, y1, x2, y2 = map(int, res.boxes.xyxy[i].tolist())
x1, y1 = max(0,x1), max(0,y1)
x2, y2 = min(W-1,x2), min(H-1,y2)

# crop & prep
crop = rgb[y1:y2, x1:x2]
crop_resz = cv2.resize(crop, (INPUT_SZ, INPUT_SZ), interpolation=cv2.INTER_LINEAR)
inp = torch.from_numpy((crop_resz.astype(np.float32)/255.).transpose(2,0,1)).unsqueeze(0).to(device).requires_grad_(True)

# get a class target from crop (still class-specific Grad-CAM)
r_crop = model_det.predict(source=crop, conf=0.01, imgsz=INPUT_SZ, verbose=False)[0]
top_cls = int(r_crop.boxes.cls[torch.argmax(r_crop.boxes.conf)].item()) if (r_crop.boxes is not None and len(r_crop.boxes)>0) else 0

class YoloRowTarget:
    def __init__(self, cls_idx, model_ref):
        self.cls_idx = int(cls_idx); self.nc = model_ref.model.model[-1].nc
    def __call__(self, outputs):
        pred = outputs[0]
        if pred.size(1) > 4 + self.nc:
            obj = pred[:,4].sigmoid(); cls = pred[:,5 + self.cls_idx].sigmoid(); return (obj*cls).max()
        else:
            cls = pred[:,4 + self.cls_idx].sigmoid(); return cls.max()

target = YoloRowTarget(top_cls, model_cam)
model_cam.model.zero_grad(set_to_none=True)

# Grad-CAM (API without use_cuda)
gcam = GradCAM(model=model_cam.model, target_layers=target_layers)
cam_map = gcam(input_tensor=inp, targets=[target])[0]
gcam.activations_and_grads.release(); del gcam

# overlay (mask to top-k to avoid big purple slabs)
thr = np.quantile(cam_map, 1 - TOPK)
mask = (cam_map >= thr).astype(np.float32)
cam_map = pct_norm(cam_map, LO_PCT, HI_PCT)

heat = cv2.applyColorMap((cam_map*255).astype(np.uint8), cv2.COLORMAP_JET)
heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB).astype(np.float32)
heat = heat * mask[..., None]  # sparsify

blend = np.clip(HEAT_W*heat + IMG_W*crop_resz.astype(np.float32), 0, 255).astype(np.uint8)

# paste back into full image
out = rgb.copy()
out[y1:y2, x1:x2] = cv2.resize(blend, (x2 - x1, y2 - y1), interpolation=cv2.INTER_LINEAR)

os.makedirs("/content/gradcam", exist_ok=True)
save_path = f"/content/gradcam/{os.path.basename(IMG_PATH).replace('.jpg','_gradcam.jpg')}"
imwrite_rgb(save_path, out)
print("Saved:", save_path)


Saved: /content/gradcam/Weevil-102-_jpg.rf.34c74f7621cdea247c335cac46f5c73f_gradcam.jpg
